# Jointure des tables Contrat et Sinistre

Bienvenue dans la partie jointure. Après avoir nettoyé les tables Contrat et Sinistre, je vais les joindre pour pouvoir commencer l'analyse. Pour la jointure, je décide d'abord de modifier la table Contrat. En effet, cette table contient 9 lignes qui comportent ou non un identifiant sinistre. Je décide donc de prendre la table Contrat en multipliant chaque numéro de contrat par le nombre de sinistres qui lui sont reliés (ex un contrat ayant 2 sinitres aura 2 lignes) et comme la table Contrat conserve aussi l'historique des contrats, je ne garde que la ligne concernée par un sinistre. Enfin, je fais un inner join entre ces deux tables pour n'avoir que les contrats ayant eu un sinistre, et seulement des sinistres reliés à un contrat.

In [1]:
import pandas as pd

cols_sinistres = [
    'id1_AssBase', 'id1_Ass0km', 'id1_AssVHR', 
    'id2_AssBase', 'id2_Ass0km', 'id2_AssVHR', 
    'id3_AssBase', 'id3_Ass0km', 'id3_AssVHR'
]
dtype_dict = {col: str for col in cols_sinistres}

contrat = pd.read_csv('contrat_cleaned.csv', sep=';', encoding='utf-8-sig', dtype=dtype_dict)
sinistre = pd.read_csv('sinistre_cleaned.csv', sep=';', encoding='utf-8-sig')

contrat['date_debut'] = pd.to_datetime(contrat['date_debut'])
contrat['date_fin'] = pd.to_datetime(contrat['date_fin'])
sinistre['date_survenance_sinistre'] = pd.to_datetime(sinistre['date_survenance_sinistre'])

mask_a_un_sinistre = contrat[cols_sinistres].notna().any(axis=1)
contrat_sinistre = contrat[mask_a_un_sinistre].copy()

cols_fixes = [c for c in contrat_sinistre.columns if c not in cols_sinistres]
contrat_long = pd.melt(
    contrat_sinistre, 
    id_vars=cols_fixes, 
    value_vars=cols_sinistres, 
    var_name='source_garantie', 
    value_name='identifiant_sinistre'
).dropna(subset=['identifiant_sinistre'])

base_sinistres_pure = pd.merge(contrat_long, sinistre, on='identifiant_sinistre', how='inner')

mask_temporel = (
    (base_sinistres_pure['date_survenance_sinistre'] >= base_sinistres_pure['date_debut']) & 
    (base_sinistres_pure['date_survenance_sinistre'] <= base_sinistres_pure['date_fin'])
)
base_analyse_sin = base_sinistres_pure[mask_temporel].copy()

base_analyse_sin = base_analyse_sin.sort_values('date_debut').drop_duplicates(subset=['identifiant_sinistre'], keep='last')

print(f"Base de sinistralité prête : {len(base_analyse_sin)} lignes.")
print(f"Montant total vérifié : {base_analyse_sin['montant_reglement'].sum():.2f} €")

Base de sinistralité prête : 28945 lignes.
Montant total vérifié : 5541009.51 €


Voici maintenant un aperçu de la jointure :

In [2]:
base_analyse_sin.head(5)

,identifiant_contrat,annee_exercice,immatriculation_vehicule,date_debut,date_fin,exposition,age_conducteur,genre_conducteur,type_permis,anciennete_permis,...,cotisation_vhr,source_garantie,identifiant_sinistre,garantie_sinistre,date_survenance_sinistre,date_declaration_sinistre,date_cloture_sinistre,gest_sin,montant_evaluation,montant_reglement
19166,C002919456,2019,JT-1368-MM,2019-01-01,2019-12-31,1.0,41.15,Homme,Traditionnel,19.55,...,0.00,id1_AssBase,S19-0583771,Assistance de base,2019-03-26,2019-03-27,2019-04-21,2019-04-21,180.14,180.14
19175,C002919866,2019,ZL-4429-JR,2019-01-01,2019-12-31,1.0,62.23,Femme,Traditionnel,41.29,...,0.00,id1_AssBase,S19-0584246,Assistance de base,2019-04-25,2019-04-26,2019-05-21,2019-05-21,180.44,180.44
19178,C002919935,2019,BM-9154-RS,2019-01-01,2019-12-31,1.0,41.70,Homme,Traditionnel,23.37,...,0.00,id1_AssBase,S19-0584323,Assistance de base,2019-08-02,2019-08-05,2019-08-30,2019-08-30,162.01,162.01
19180,C002919964,2019,PM-8659-KX,2019-01-01,2019-12-31,1.0,31.79,Femme,Traditionnel,11.53,...,0.00,id1_AssBase,S19-0584359,Assistance de base,2019-07-10,2019-07-10,2019-07-10,2019-07-10,194.15,194.15
52425,C003297596,2019,HD-9121-JN,2019-01-01,2019-12-31,1.0,46.81,Femme,Traditionnel,26.18,...,19.32,id1_Ass0km,S19-2361120,Assistance 0 km,2019-07-19,2019-07-27,NaN,2019-07-27,150.00,0.00


Je garde donc cette table qui va servir pour l'analyse :

In [3]:
base_analyse_sin.to_csv('base_sinistralite_pure.csv', index=False, sep=';', encoding='utf-8-sig')
print("Fichier sauvegardé avec succès dans votre répertoire de travail !")

Fichier sauvegardé avec succès dans votre répertoire de travail !
